In [2]:
import bz2
import pandas as pd
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from tqdm import tqdm
import shutil
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
import pandas as pd
import re

# Load dataset
df = pd.read_csv(
    "test.ft.txt",
    sep="\t",
    header=None,
    names=["text"],
    engine="python",
    encoding="latin-1"
)

# Remove labels
df["review"] = df["text"].str.replace("__label__1 ", "", regex=False)
df["review"] = df["review"].str.replace("__label__2 ", "", regex=False)


print(df.head())

                                                text  \
0  __label__2 Great CD: My lovely Pat has one of ...   
1  __label__2 One of the best game music soundtra...   
2  __label__1 Batteries died within a year ...: I...   
3  __label__2 works fine, but Maha Energy is bett...   
4  __label__2 Great for the non-audiophile: Revie...   

                                              review  
0  Great CD: My lovely Pat has one of the GREAT v...  
1  One of the best game music soundtracks - for a...  
2  Batteries died within a year ...: I bought thi...  
3  works fine, but Maha Energy is better: Check o...  
4  Great for the non-audiophile: Reviewed quite a...  


In [10]:

# -----------------------------
# Custom Regex Tokenizer
# -----------------------------

def custom_tokenizer(text):

    # Regex patterns
    patterns = [
        r'[$₹€£¥]\s?\d+(?:,\d{3})*(?:\.\d+)?',      # Currency
        r'\b\d+(?:\.\d+)?\s?(?:GB|TB|MB|GHz|MHz|W|V|mAh|inch|cm|mm|kg|g|fps|Hz|MP)\b',  # Tech specs
        r'\b\d+p\b',                               # 1080p, 720p
        r'\b[a-zA-Z]+\d+\b',                       # RTX4090, iPhone15
        r'\b\d+\.\d+\b',                           # Decimal numbers
        r'\b\w+\b'                                 # Normal words
    ]

    combined_pattern = '|'.join(patterns)

    tokens = re.findall(combined_pattern, str(text), flags=re.IGNORECASE)

    return tokens

# Apply tokenizer
df["tokens"] = df["review"].apply(custom_tokenizer)

# Show results
print(df[["review", "tokens"]].head())

                                              review  \
0  Great CD: My lovely Pat has one of the GREAT v...   
1  One of the best game music soundtracks - for a...   
2  Batteries died within a year ...: I bought thi...   
3  works fine, but Maha Energy is better: Check o...   
4  Great for the non-audiophile: Reviewed quite a...   

                                              tokens  
0  [Great, CD, My, lovely, Pat, has, one, of, the...  
1  [One, of, the, best, game, music, soundtracks,...  
2  [Batteries, died, within, a, year, I, bought, ...  
3  [works, fine, but, Maha, Energy, is, better, C...  
4  [Great, for, the, non, audiophile, Reviewed, q...  
